# Improved LLM Plans
Original feature, Only one mental Keggle Mental health information used.
Then, based on information, how should I imporve my model,
In inital Training, gpt model4, and we have serveral information inside.
Next approch, I like to Hugging Datasets and Mental health datasets

Currently, feature we just populate randomly.
Next what, I want is text related llm topic clustering,
Based on token, clustering.


In [4]:
# Hugging Datasets called
raw_data_path = "raw_data"
cleaned_data_path= "cleaned_data"



In [29]:
import pandas as pd
# Inital Raw Data Used.

def clean_keggle_df(data_path):
    df = pd.read_csv(data_path)
    # print(keggle_df.columns)
    df = df[["questionText", "topics", "re_diagnosis","clean_answer_text"]]
    # Lower case
    df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    # remove non-world
    df = keggle_df.replace(to_replace=r'[^\w\s]', value="", regex=True)
    # remove number
    df = df.replace(to_replace=r'\d', value='', regex=True)

    return df

def clean_hugging_df(data_path):
    df = pd.read_csv(data_path)

    df = df[["questionTitle", "questionText", "topic", "answerText"]]
    # Lower case
    df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    # remove non-world
    df["questionText"] = df["questionTitle"].fillna('') + " " + df["questionText"].fillna('')
    df = df.replace(to_replace=r'[^\w\s]', value="", regex=True)
    # remove number
    df = df.replace(to_replace=r'\d', value='', regex=True)
    df = df[["questionText", "topic", "answerText"]]

    # print(hugging_df.head)
    return df

keggle_df = clean_keggle_df(f"{raw_data_path}/counsel_cleaned.csv")   
print(keggle_df.shape)
keggle_df.to_csv(f"{cleaned_data_path}/cleaned_counsel.csv")
hugging_df = clean_hugging_df(f"{raw_data_path}/huggin_counsel_chat.csv")
print(hugging_df.shape)
hugging_df

(1373, 4)
(2775, 3)


/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_83960/436549379.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_83960/436549379.py:22: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


,questionText,topic,answerText
0,do i have too many issues for counseling i hav...,depression,it is very common for people to have multiple ...
1,do i have too many issues for counseling i hav...,depression,ive never heard of someone having too many iss...
2,do i have too many issues for counseling i hav...,depression,absolutely not i strongly recommending workin...
3,do i have too many issues for counseling i hav...,depression,let me start by saying there are never too man...
4,do i have too many issues for counseling i hav...,depression,i just want to acknowledge you for the courage...
...,...,...,...
2770,are some clients more difficult than others wh...,counselingfundamentals,although many clients have the capacity to be ...
2771,are some clients more difficult than others wh...,counselingfundamentals,i usually dont label a client as difficult bec...
2772,are some clients more difficult than others wh...,counselingfundamentals,dang right heh heh and correct me if im wrong...
2773,are some clients more difficult than others wh...,counselingfundamentals,yes just like some relationships outside of ou...


In [37]:
import pandas as pd

# 데이터 파일 경로 설정

# 파일 읽기
hugging_df = pd.read_csv(f"{cleaned_data_path}/cleaned_hugging.csv")
counsel_df = pd.read_csv(f"{cleaned_data_path}/cleaned_counsel.csv")

# 칼럼 이름 변경
# 칼럼 이름 변경 확인
hugging_df.rename(columns={"questionText": "question_text",
                           "topic": "topics", 
                           "answerText": "answer_text"}, inplace=True)
hugging_df.drop(columns=['Unnamed: 0'], inplace=True)

print("After renaming:", hugging_df.columns)
counsel_df.drop(columns=['Unnamed: 0', 're_diagnosis'], inplace=True)  # 're_diagnosis'도 사용하지 않을 경우 제거
print(counsel_df.columns)
counsel_df.rename(columns={"questionText": "question_text", 
                           "clean_answer_text": "answer_text"}, inplace=True)

hugging_df = hugging_df[["question_text", "topics", "answer_text"]]
counsel_df = counsel_df[["question_text", "topics", "answer_text"]]
# 필요한 칼럼만 선택
print(hugging_df.columns)
print(counsel_df.columns)

# 데이터 결합
combined_dataset = pd.concat([hugging_df, counsel_df], ignore_index=True)

# 칼럼 확인
print(combined_dataset.columns)
combined_dataset.to_csv(f"{cleaned_data_path}/combined_output.csv")


After renaming: Index(['questionTitle', 'question_text', 'topics', 'answer_text'], dtype='object')
Index(['questionText', 'topics', 'clean_answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')


In [39]:
# Chat Promt, Design.
combined_dataset = pd.read_csv(f"{cleaned_data_path}/combined_output.csv")
print(combined_dataset.columns)



Index(['Unnamed: 0', 'question_text', 'topics', 'answer_text'], dtype='object')


In [67]:
# NLTK test
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
text = "This is an example. Here is another sentence."
sentences = sent_tokenize(text)

print(sentences)


[nltk_data] Downloading package punkt_tab to /Users/yoon/nltk_data...


['This is an example.', 'Here is another sentence.']


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [70]:
# Cleaned Combined Datasets too shorts and too long
import re
from nltk.tokenize import word_tokenize

def is_noisy(text: str) -> bool:
    # 한글이나 영문 글자가 하나도 없는 경우 노이즈로 간주
    if re.search(r'[가-힣A-Za-z]', text) is None:
        return True
    # 문자와 숫자를 제외한 특수문자가 과도하게 많은 경우 노이즈로 간주
    cleaned = re.sub(r'[^\w\s]', '', text) 
    if len(cleaned) == 0 or len(cleaned) < len(text) * 0.02:  
        return True
    return False


def clean_combined_dataset(df):
    def token_count(text):
        tokens = sent_tokenize(text)
        return len(tokens)
    print(df.shape)
    # Apply the token_count function to calculate the number of tokens in questions and answers
    df = df.dropna()
    df['q_token_count'] = df['question_text'].apply(token_count)
    df['a_token_count'] = df['answer_text'].apply(token_count)
    print(df.shape)
    print(df.head)
    # Filter rows where token count is less than 5 or more than 500
    df = df[(df['q_token_count'] >= 5) & (df['q_token_count'] < 500)]
    df = df[(df['a_token_count'] >= 5) & (df['a_token_count'] < 500)]
    

    df = df.drop_duplicates(subset='question_text', keep='first')
    df = df.drop_duplicates(subset='answer_text', keep='first')

    df = df[~df['question_text'].apply(is_noisy)]
    df = df[~df['answer_text'].apply(is_noisy)]





    return df

clean_combined_dataset(combined_dataset)

(4148, 4)
(3985, 6)
<bound method NDFrame.head of       Unnamed: 0                                      question_text  \
0              0  i have so many issues to address i have a hist...   
1              1  i have so many issues to address i have a hist...   
2              2  i have so many issues to address i have a hist...   
3              3  i have so many issues to address i have a hist...   
4              4  i have so many issues to address i have a hist...   
...          ...                                                ...   
4143        4143  my grandsons stepmother sends him to school wi...   
4144        4144  my boyfriend is in recovery from drug addictio...   
4145        4145  the birth mother attempted suicide several tim...   
4146        4146  i think adult life is making him depressed and...   
4147        4147  i just took a job that requires me to travel f...   

                                             topics  \
0                                        d

/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_83960/3169828074.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['q_token_count'] = df['question_text'].apply(token_count)
/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_83960/3169828074.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['a_token_count'] = df['answer_text'].apply(token_count)


KeyError: 'answer_text'